<a href="https://colab.research.google.com/github/landelaniblessing-cyber/WEEK3-HEALTHCARE-DATA-SCIENCE/blob/main/notebooks/Week3_Healthcare_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import the libraries needed for data analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print("Libraries imported successfully!")

In [ ]:
# Load all six hospital datasets

import pandas as pd
import os

encounters = pd.read_csv('/content/encounters.csv')
patients = pd.read_csv('/content/patients.csv')
procedures = pd.read_csv('/content/procedures.csv')
payers = pd.read_csv('/content/payers.csv')
organizations = pd.read_csv('/content/organizations.csv')
dictionary = pd.read_csv('/content/data_dictionary.csv')

print("All six datasets loaded successfully!")

In [ ]:
# Check the size of each dataset

datasets = {
    "Encounters": encounters,
    "Patients": patients,
    "Procedures": procedures,
    "Payers": payers,
    "Organizations": organizations,
    "Data Dictionary": dictionary
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows and {df.shape[1]} columns")

Encounters: 1000 rows and 15 columns
Patients: 500 rows and 10 columns
Procedures: 800 rows and 8 columns
...

In [ ]:
# Display column names in all six datasets

for name, df in datasets.items():
    print("\n", "=" * 50)
    print(name.upper())
    print("=" * 50)
    print(df.columns.tolist())

In [ ]:
for name, df in datasets.items():
    print("\n" + name.upper())
    print(", ".join(df.columns))

ENCOUNTERS
id, start, stop, patient, ...

In [ ]:
print(encounters.columns.tolist())

In [ ]:
# Identify the different encounter classes

print(encounters['ENCOUNTERCLASS'].value_counts())

In [ ]:
# Check missing values in important columns

important_columns = [
    'Id',
    'ENCOUNTERCLASS',
    'BASE_ENCOUNTER_COST',
    'TOTAL_CLAIM_COST'
]

print("MISSING VALUES:")
print(encounters[important_columns].isnull().sum())

print("\nDUPLICATE ENCOUNTER IDs:")
print(encounters['Id'].duplicated().sum())

print("\nDATA TYPES:")
print(encounters[important_columns].dtypes)

In [ ]:
# Summary statistics for encounter costs

print(encounters[
    ['BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST']
].describe().round(2))

In [ ]:
# Visualise the distribution of encounter costs

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(encounters['BASE_ENCOUNTER_COST'], bins=30, kde=True)
plt.title('Distribution of Base Encounter Costs')
plt.xlabel('Base Encounter Cost')

plt.subplot(1, 2, 2)
sns.histplot(encounters['TOTAL_CLAIM_COST'], bins=30, kde=True)
plt.title('Distribution of Total Claim Costs')
plt.xlabel('Total Claim Cost')

plt.tight_layout()
plt.show()

In [ ]:
# Investigate zero and unusually high total claim costs

print("Number of zero-cost encounters:")
print((encounters['TOTAL_CLAIM_COST'] == 0).sum())

print("\nNumber of negative-cost encounters:")
print((encounters['TOTAL_CLAIM_COST'] < 0).sum())

print("\nHighest 10 total claim costs:")
print(
    encounters.nlargest(10, 'TOTAL_CLAIM_COST')[
        ['ENCOUNTERCLASS', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST']
    ].to_string(index=False)
)

In [ ]:
# Compare costs across encounter classes

cost_comparison = encounters.groupby('ENCOUNTERCLASS').agg(
    Number_of_Encounters=('Id', 'count'),
    Average_Base_Cost=('BASE_ENCOUNTER_COST', 'mean'),
    Median_Base_Cost=('BASE_ENCOUNTER_COST', 'median'),
    Average_Total_Claim_Cost=('TOTAL_CLAIM_COST', 'mean'),
    Median_Total_Claim_Cost=('TOTAL_CLAIM_COST', 'median')
).round(2)

print(cost_comparison)

In [ ]:
# Average total claim cost by encounter class

plt.figure(figsize=(10, 6))

sns.barplot(
    data=encounters,
    x='ENCOUNTERCLASS',
    y='TOTAL_CLAIM_COST',
    estimator='mean',
    errorbar=None
)

plt.title('Average Total Claim Cost by Encounter Class')
plt.xlabel('Encounter Class')
plt.ylabel('Average Total Claim Cost')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Kruskal-Wallis statistical test

groups = [
    group['TOTAL_CLAIM_COST'].values
    for name, group in encounters.groupby('ENCOUNTERCLASS')
]

statistic, p_value = stats.kruskal(*groups)

print("Kruskal-Wallis Test Results")
print("---------------------------")
print("Test statistic:", round(statistic, 4))
print("P-value:", p_value)

if p_value < 0.05:
    print("Conclusion: Reject the null hypothesis.")
    print("There is evidence of a statistically significant difference.")
else:
    print("Conclusion: Fail to reject the null hypothesis.")
    print("There is insufficient evidence of a statistically significant difference.")

In [ ]:
from itertools import combinations
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Compare every pair of encounter classes

classes = encounters['ENCOUNTERCLASS'].unique()

results = []

for class1, class2 in combinations(classes, 2):
    group1 = encounters.loc[
        encounters['ENCOUNTERCLASS'] == class1,
        'TOTAL_CLAIM_COST'
    ]

    group2 = encounters.loc[
        encounters['ENCOUNTERCLASS'] == class2,
        'TOTAL_CLAIM_COST'
    ]

    statistic, p_value = mannwhitneyu(
        group1, group2, alternative='two-sided'
    )

    results.append({
        'Class 1': class1,
        'Class 2': class2,
        'U statistic': statistic,
        'Raw p-value': p_value
    })

pairwise_results = pd.DataFrame(results)

# Apply Bonferroni correction
pairwise_results['Adjusted p-value'] = multipletests(
    pairwise_results['Raw p-value'],
    method='bonferroni'
)[1]

pairwise_results['Significant'] = (
    pairwise_results['Adjusted p-value'] < 0.05
)

print(pairwise_results.round(6).to_string(index=False))

In [ ]:
# Compare total claim cost distributions by encounter class

plt.figure(figsize=(12, 7))

sns.boxplot(
    data=encounters,
    x='ENCOUNTERCLASS',
    y='TOTAL_CLAIM_COST'
)

plt.title('Distribution of Total Claim Costs by Encounter Class')
plt.xlabel('Encounter Class')
plt.ylabel('Total Claim Cost')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Logarithmic boxplot of total claim costs

plt.figure(figsize=(12, 7))

sns.boxplot(
    data=encounters,
    x='ENCOUNTERCLASS',
    y='TOTAL_CLAIM_COST'
)

plt.yscale('symlog', linthresh=1)

plt.title('Total Claim Costs by Encounter Class (Logarithmic Scale)')
plt.xlabel('Encounter Class')
plt.ylabel('Total Claim Cost (Log Scale)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Check repeated patient encounters

print("Total encounters:", len(encounters))

print("Unique patients:", encounters['PATIENT'].nunique())

print("Patients with multiple encounters:",
      (encounters['PATIENT'].value_counts() > 1).sum())

print("\nAverage encounters per patient:",
      round(len(encounters) / encounters['PATIENT'].nunique(), 2))

In [ ]:
patient_class_counts = encounters.groupby('PATIENT')['ENCOUNTERCLASS'].nunique()

print("Patients appearing in multiple encounter classes:",
      (patient_class_counts > 1).sum())

print("\nPatients appearing in only one encounter class:",
      (patient_class_counts == 1).sum())

In [ ]:
patient_class_summary = (
    encounters
    .groupby(['PATIENT', 'ENCOUNTERCLASS'])['TOTAL_CLAIM_COST']
    .mean()
    .reset_index()
)

print(patient_class_summary.head(10))

In [ ]:
patient_class_counts = patient_class_summary.groupby('PATIENT')['ENCOUNTERCLASS'].nunique()

print("Patients with all six encounter classes:",
      (patient_class_counts == 6).sum())

print("\nPatients with fewer than six encounter classes:",
      (patient_class_counts < 6).sum())

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

model_data = encounters[
    ['PATIENT', 'ENCOUNTERCLASS', 'TOTAL_CLAIM_COST']
].copy()

model_data['LOG_CLAIM_COST'] = np.log1p(
    model_data['TOTAL_CLAIM_COST']
)

print(model_data.head())
print("\nNumber of observations:", len(model_data))
print("Number of patients:", model_data['PATIENT'].nunique())

In [ ]:
mixed_model = smf.mixedlm(
    "LOG_CLAIM_COST ~ C(ENCOUNTERCLASS)",
    data=model_data,
    groups=model_data["PATIENT"]
)

mixed_result = mixed_model.fit(
    method="lbfgs",
    reml=True
)

print(mixed_result.summary())

In [ ]:
patient_encounter_counts = encounters['PATIENT'].value_counts()

print("Patients with the most encounters:")
print(patient_encounter_counts.head(10))

print("\nPatient encounter count summary:")
print(patient_encounter_counts.describe())

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

gee_model = smf.gee(
    "LOG_CLAIM_COST ~ C(ENCOUNTERCLASS)",
    groups="PATIENT",
    data=model_data,
    family=sm.families.Gaussian(),
    cov_struct=sm.cov_struct.Exchangeable()
)

gee_result = gee_model.fit()

print(gee_result.summary())

In [ ]:
import numpy as np

coefficients = gee_result.params

percentage_effects = (
    (np.exp(coefficients.drop('Intercept')) - 1) * 100
)

print("Percentage differences compared with ambulatory:")
print(percentage_effects.round(2))

In [ ]:
print("Estimated within-patient correlation:",
      gee_result.cov_struct.dep_params)

In [ ]:
# Calculate the 99th percentile of encounters per patient
cutoff = patient_encounter_counts.quantile(0.99)

print("99th percentile cutoff:", cutoff)

print("Patients above the cutoff:",
      (patient_encounter_counts > cutoff).sum())

print("Encounters belonging to these patients:",
      patient_encounter_counts[patient_encounter_counts > cutoff].sum())

In [ ]:
# Identify patients with 274 or fewer encounters
regular_patients = patient_encounter_counts[
    patient_encounter_counts <= cutoff
].index

# Create a temporary dataset excluding the 10 highest-frequency patients
sensitivity_data = model_data[
    model_data['PATIENT'].isin(regular_patients)
].copy()

print("Original observations:", len(model_data))
print("Observations after exclusion:", len(sensitivity_data))
print("Patients remaining:", sensitivity_data['PATIENT'].nunique())

In [ ]:
gee_sensitivity = smf.gee(
    "LOG_CLAIM_COST ~ C(ENCOUNTERCLASS)",
    groups="PATIENT",
    data=sensitivity_data,
    family=sm.families.Gaussian(),
    cov_struct=sm.cov_struct.Exchangeable()
)

gee_sensitivity_result = gee_sensitivity.fit()

print(gee_sensitivity_result.summary())

In [ ]:
comparison = pd.DataFrame({
    'Original Coefficient': gee_result.params,
    'Original P-value': gee_result.pvalues,
    'Sensitivity Coefficient': gee_sensitivity_result.params,
    'Sensitivity P-value': gee_sensitivity_result.pvalues
})

print(comparison.round(4))

In [ ]:
comparison.to_csv(
    'GEE_Model_Comparison.csv',
    index=True
)

In [ ]:
# Save descriptive statistics by encounter class
class_summary.to_csv(
    'Encounter_Class_Descriptive_Statistics.csv',
    index=False
)

# Save pairwise Mann-Whitney test results
pairwise_results.to_csv(
    'Pairwise_Statistical_Tests.csv',
    index=False
)

# Save the GEE model results
gee_results_table = pd.DataFrame({
    'Coefficient': gee_result.params,
    'Standard_Error': gee_result.bse,
    'P_Value': gee_result.pvalues
})

gee_results_table.to_csv(
    'GEE_Main_Results.csv'
)

print("All results saved successfully!")

In [ ]:
class_summary = (
    encounters.groupby('ENCOUNTERCLASS')
    .agg(
        Number_of_Encounters=('TOTAL_CLAIM_COST', 'count'),
        Average_Base_Cost=('BASE_ENCOUNTER_COST', 'mean'),
        Median_Base_Cost=('BASE_ENCOUNTER_COST', 'median'),
        Average_Total_Claim_Cost=('TOTAL_CLAIM_COST', 'mean'),
        Median_Total_Claim_Cost=('TOTAL_CLAIM_COST', 'median')
    )
    .reset_index()
)

print(class_summary.round(2))

In [ ]:
class_summary.to_csv(
    'Encounter_Class_Descriptive_Statistics.csv',
    index=False
)

pairwise_results.to_csv(
    'Pairwise_Statistical_Tests.csv',
    index=False
)

gee_results_table = pd.DataFrame({
    'Coefficient': gee_result.params,
    'Standard_Error': gee_result.bse,
    'P_Value': gee_result.pvalues
})

gee_results_table.to_csv('GEE_Main_Results.csv')

print("All results saved successfully!")

In [ ]:
import os

os.makedirs('visualisations', exist_ok=True)

print("Visualisations folder ready!")

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=encounters,
    x='BASE_ENCOUNTER_COST',
    bins=30,
    kde=True
)

plt.title('Distribution of Base Encounter Costs')
plt.xlabel('Base Encounter Cost')
plt.ylabel('Frequency')

plt.tight_layout()

plt.savefig(
    'visualisations/01_Base_Encounter_Cost_Distribution.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=encounters,
    x='TOTAL_CLAIM_COST',
    bins=50,
    kde=True
)

plt.title('Distribution of Total Claim Costs')
plt.xlabel('Total Claim Cost')
plt.ylabel('Frequency')

plt.tight_layout()

plt.savefig(
    'visualisations/02_Total_Claim_Cost_Distribution.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=class_summary,
    x='ENCOUNTERCLASS',
    y='Average_Total_Claim_Cost'
)

plt.title('Average Total Claim Cost by Encounter Class')
plt.xlabel('Encounter Class')
plt.ylabel('Average Total Claim Cost')
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    'visualisations/03_Average_Claim_Cost_by_Class.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

sns.boxplot(
    data=encounters,
    x='ENCOUNTERCLASS',
    y='TOTAL_CLAIM_COST'
)

plt.title('Total Claim Costs by Encounter Class')
plt.xlabel('Encounter Class')
plt.ylabel('Total Claim Cost')
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    'visualisations/04_Claim_Cost_Boxplot.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

sns.boxplot(
    data=encounters,
    x='ENCOUNTERCLASS',
    y='TOTAL_CLAIM_COST'
)

plt.yscale('symlog', linthresh=1)

plt.title('Total Claim Costs by Encounter Class (Logarithmic Scale)')
plt.xlabel('Encounter Class')
plt.ylabel('Total Claim Cost (Symmetric Log Scale)')
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    'visualisations/05_Claim_Cost_Boxplot_Log_Scale.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
print(os.listdir('visualisations'))

In [ ]:
import shutil

shutil.make_archive(
    'week3_visualisations',
    'zip',
    'visualisations'
)

print("All five graphs have been packaged successfully!")

In [ ]:
from google.colab import files

files.download('week3_visualisations.zip')

In [ ]:
import shutil
import os

result_files = [
    'Encounter_Class_Descriptive_Statistics.csv',
    'Pairwise_Statistical_Tests.csv',
    'GEE_Main_Results.csv',
    'GEE_Model_Comparison.csv'
]

for file in result_files:
    print(file, "—", "Found" if os.path.exists(file) else "Missing")

In [ ]:
import zipfile

with zipfile.ZipFile('week3_statistical_results.zip', 'w') as zipf:
    for file in result_files:
        zipf.write(file)

print("All four statistical results have been packaged successfully!")

In [ ]:
from google.colab import files

files.download('week3_statistical_results.zip')